# Why Ensembles Work
From-scratch NumPy simulations of the three ideas behind every ensemble:

1. **Majority vote of independent classifiers** (exact binomial formula + Monte Carlo).
2. **Error decorrelation**: correlated mistakes destroy the benefit of voting.
3. **Bias / variance of averaged models**: `Var = rho*s^2 + (1-rho)*s^2/M`;
   averaging kills variance but leaves bias untouched.

Plots are saved to `plots/`.

In [ ]:
import os
import warnings
from math import comb

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.simplefilter("ignore")
os.makedirs("plots", exist_ok=True)
rng = np.random.default_rng(0)
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "figure.dpi": 110})

## 1. Exact majority-vote accuracy
For N independent classifiers with accuracy p, the vote is right when more than N/2 are right:

`P = sum_{k > N/2} C(N,k) p^k (1-p)^(N-k)`

In [ ]:
def majority_accuracy(N, p):
    """Exact probability that a strict majority of N independent classifiers is correct."""
    return sum(comb(N, k) * p**k * (1 - p) ** (N - k) for k in range(N // 2 + 1, N + 1))


for N in (1, 3, 11, 21, 51, 101):
    print(f"N={N:4d}  p=0.60 -> majority accuracy = {majority_accuracy(N, 0.6):.4f}")
# NOTE: the exact formula gives ~0.826 for N=21 and ~0.98 for N=101 at p=0.6
# (the doc's quoted 74% / 84% are conservative; the formula itself is what matters).

In [ ]:
Ns = np.arange(1, 202, 2)
fig, ax = plt.subplots(figsize=(6.5, 4))
for p in (0.51, 0.55, 0.6, 0.7):
    ax.plot(Ns, [majority_accuracy(int(n), p) for n in Ns], label=f"p={p}")
ax.set(xlabel="number of independent classifiers N", ylabel="majority-vote accuracy",
       title="Independent voters: accuracy -> 1 as N grows")
ax.legend()
fig.tight_layout(); fig.savefig("plots/01_majority_vote_theory.png"); plt.close(fig)

## 2. Monte Carlo simulation of voting
Simulate 20,000 test examples. Each classifier is right with prob. p independently.
The simulated curve should sit on top of the exact formula.

In [ ]:
def simulate_vote_curve(p, rho, n_max=101, n_examples=20000, rng=rng):
    """Majority-vote accuracy for N=1,3,..,n_max with *correlated* errors.

    Correlation model (beta-binomial): every example has its own 'easiness'
    q ~ Beta(p*c, (1-p)*c); classifiers are then correct independently with prob q.
    Marginal accuracy stays p; the correlation between two classifiers' correctness
    is exactly rho = 1/(1+c).  rho=0 means independent.
    """
    if rho <= 0:
        q = np.full(n_examples, p)
    elif rho >= 1:                      # identical classifiers: all right or all wrong together
        q = (rng.random(n_examples) < p).astype(float)
    else:
        c = (1 - rho) / rho
        q = rng.beta(p * c, (1 - p) * c, size=n_examples)
    correct = rng.random((n_examples, n_max)) < q[:, None]
    cum = np.cumsum(correct, axis=1)
    Ns = np.arange(1, n_max + 1, 2)
    acc = np.array([(cum[:, n - 1] > n / 2).mean() for n in Ns])
    return Ns, acc


Ns_sim, acc_sim = simulate_vote_curve(0.6, 0.0)
exact = np.array([majority_accuracy(int(n), 0.6) for n in Ns_sim])
print("max |simulation - exact| =", np.abs(acc_sim - exact).max().round(4))

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(Ns_sim, exact, "k-", label="exact formula")
ax.plot(Ns_sim, acc_sim, "o", ms=3, label="Monte Carlo")
ax.set(xlabel="N classifiers", ylabel="majority accuracy", title="Simulation matches the binomial formula (p=0.6)")
ax.legend(); fig.tight_layout(); fig.savefig("plots/02_simulation_vs_exact.png"); plt.close(fig)

## 3. Error decorrelation: diversity is the key requirement
Keep every classifier at 60% accuracy but make their mistakes more and more alike.
With rho -> 1 all models fail on the same examples and voting gives nothing.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
for rho in (0.0, 0.05, 0.2, 0.5, 1.0):
    Ns_c, acc_c = simulate_vote_curve(0.6, rho)
    ax.plot(Ns_c, acc_c, label=f"error correlation rho={rho}")
    print(f"rho={rho:4.2f}  acc(N=1)={acc_c[0]:.3f}  acc(N=101)={acc_c[-1]:.3f}")
ax.set(xlabel="N classifiers (each 60% accurate)", ylabel="majority accuracy",
       title="Correlated errors: the ensemble stops improving")
ax.legend(); fig.tight_layout(); fig.savefig("plots/03_error_correlation.png"); plt.close(fig)

## 4. Variance of an average of M models
Each model's prediction at a fixed point x0 is
`f(x0) + bias + sqrt(rho)*s*z_shared + sqrt(1-rho)*s*e_i`.
Theory: `Var(mean) = rho*s^2 + (1-rho)*s^2/M` and `Bias(mean) = bias` (unchanged).

In [ ]:
def ensemble_stats(M, rho, sigma=1.0, bias=0.5, n_trials=20000, rng=rng):
    """Empirical (bias, variance) of the average of M correlated noisy models."""
    shared = rng.normal(size=(n_trials, 1))
    indiv = rng.normal(size=(n_trials, M))
    preds = 1.0 + bias + sigma * (np.sqrt(rho) * shared + np.sqrt(1 - rho) * indiv)
    avg = preds.mean(axis=1)
    return avg.mean() - 1.0, avg.var()


Ms = [1, 2, 5, 10, 20, 50, 100]
fig, ax = plt.subplots(figsize=(6.5, 4))
for rho in (0.0, 0.2, 0.5):
    emp = [ensemble_stats(M, rho)[1] for M in Ms]
    theory = [rho + (1 - rho) / M for M in Ms]
    ax.plot(Ms, emp, "o", ms=5, label=f"empirical rho={rho}")
    ax.plot(Ms, theory, "-", color=ax.lines[-1].get_color(), alpha=0.7)
    ax.axhline(rho, ls=":", color=ax.lines[-1].get_color(), alpha=0.5)
    print(f"rho={rho}: M=100 variance {emp[-1]:.3f} (theory {theory[-1]:.3f}); bias stays {ensemble_stats(100, rho)[0]:.2f}")
ax.set(xscale="log", xlabel="number of averaged models M", ylabel="variance of ensemble (s=1)",
       title="Var = rho*s^2 + (1-rho)*s^2/M  (dotted: floor rho*s^2)")
ax.legend(); fig.tight_layout(); fig.savefig("plots/04_variance_vs_M.png"); plt.close(fig)

## 5. Bias / variance of a real averaged model: bagged regression trees
Truth `f(x) = sin(3x)`; many independent noisy training sets. We fit
* a deep tree (depth 8: low bias / high variance), and
* a stump (depth 1: high bias / low variance),
each alone and averaged over 25 bootstrap resamples (bagging), then estimate
bias^2 and variance over the training sets.

In [ ]:
def f_true(x):
    return np.sin(3 * x)


def fit_tree(x, y, max_depth):
    """Tiny 1-D regression tree (least squares splits). Returns a nested tuple/leaf float."""
    if max_depth == 0 or x.size < 4 or np.ptp(x) == 0:
        return float(y.mean())
    o = np.argsort(x); xs, ys = x[o], y[o]
    cs, cs2 = np.cumsum(ys), np.arange(1, xs.size + 1)
    left_sse_gain = cs[:-1] ** 2 / cs2[:-1] + (cs[-1] - cs[:-1]) ** 2 / (xs.size - cs2[:-1])
    left_sse_gain[xs[:-1] == xs[1:]] = -np.inf
    i = int(np.argmax(left_sse_gain))
    if not np.isfinite(left_sse_gain[i]):
        return float(y.mean())
    t = 0.5 * (xs[i] + xs[i + 1])
    return (t, fit_tree(xs[:i + 1], ys[:i + 1], max_depth - 1), fit_tree(xs[i + 1:], ys[i + 1:], max_depth - 1))


def tree_eval(node, xq):
    if not isinstance(node, tuple):
        return np.full(xq.shape, node)
    t, l, r = node
    return np.where(xq <= t, tree_eval(l, xq), tree_eval(r, xq))


def bias_variance(depth, n_bags, n_datasets=200, n_train=40, noise=0.3, rng=rng):
    """Bias^2 and variance (averaged over a test grid) of a depth-limited tree, bagged or not."""
    xt = np.linspace(-1, 1, 60)
    preds = np.zeros((n_datasets, xt.size))
    for d in range(n_datasets):
        x = rng.uniform(-1, 1, n_train)
        y = f_true(x) + noise * rng.normal(size=n_train)
        if n_bags == 1:
            preds[d] = tree_eval(fit_tree(x, y, depth), xt)
        else:
            acc = np.zeros_like(xt)
            for _ in range(n_bags):
                idx = rng.integers(0, n_train, n_train)         # bootstrap
                acc += tree_eval(fit_tree(x[idx], y[idx], depth), xt)
            preds[d] = acc / n_bags
    mean_pred = preds.mean(axis=0)
    return ((mean_pred - f_true(xt)) ** 2).mean(), preds.var(axis=0).mean(), noise**2


rows = []
for deg in (8, 1):
    for bags in (1, 25):
        b2, v, n2 = bias_variance(deg, bags)
        rows.append((deg, bags, b2, v))
        print(f"depth={deg} bags={bags:2d}: bias^2={b2:.4f} variance={v:.4f} total(expected err - noise)={b2+v:.4f}")

labels = ["deep tree\nsingle", "deep tree\nbagged", "stump\nsingle", "stump\nbagged"]
fig, ax = plt.subplots(figsize=(6.5, 4))
xs = np.arange(4)
ax.bar(xs, [r[2] for r in rows], label="bias^2")
ax.bar(xs, [r[3] for r in rows], bottom=[r[2] for r in rows], label="variance")
ax.set_xticks(xs); ax.set_xticklabels(labels)
ax.set(ylabel="error components", title="Bagging cuts variance, not bias")
ax.legend(); fig.tight_layout(); fig.savefig("plots/05_bias_variance_bagging.png"); plt.close(fig)

## 6. Summary
* Independent voters with p>0.5 -> accuracy -> 1 as N grows.
* Correlated voters -> the gain is capped; diversity (bagging, feature subsets,
  boosting, different model families) is what makes ensembles useful.
* Averaging leaves bias unchanged and reduces variance down to a floor `rho*s^2`.
* So: average high-variance models (bagging), and use sequential correction (boosting) for high-bias ones.